# P1-A8: Semantic Relationships and Retrieval

### Embedding Models

#### Small (~384 dimensions)
BAAI/bge-small-en-v1.5
https://huggingface.co/BAAI/bge-small-en-v1.5

#### Medium (~768 dimensions)
prdev/mini-gte
https://huggingface.co/spaces/mteb/leaderboard

#### Large (1024-1536 dimensions)
BAAI/bge-large-en
https://huggingface.co/spaces/mteb/leaderboard

#### Model from previous assignment
Qwen/Qwen2.5-1.5B-Instruct
https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct


In [12]:
# Small
import numpy as np
from transformers import pipeline

pipe = pipeline("feature-extraction", model="BAAI/bge-small-en-v1.5")
output = np.array(pipe("test sentence"))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
print(output.shape) # 384 dimensions

(1, 4, 384)


In [14]:
# Medium
from sentence_transformers import SentenceTransformer

med_output = SentenceTransformer("prdev/mini-gte")
print(med_output.get_sentence_embedding_dimension()) # 768 dimensions

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

768


In [15]:
# Large
large_output = SentenceTransformer("BAAI/bge-large-en")
print(large_output.get_sentence_embedding_dimension()) # 1024 dimensions


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


1024


# RAG Pipeline


#### Knowledge Base

I wasn't sure how to find a social / activites dataset so I searched the web for inputs on a wide variety of topics and social settings

In [16]:
import pandas as pd
df = pd.read_csv("knowledge_base_clean.csv")
df.sample(n=3)

,id,topic,source,title,url,text,words,notes
37,36,social_events,Crafty Chica,NaN,NaN,This is a class from the Crafty Chica art retr...,164,NaN
35,39,fishing,Fish and Wildlife Conservatory,NaN,NaN,Respecting the catch is an important conservat...,169,NaN
15,19,board_games,Reddit,NaN,NaN,I think it’s a good idea for you to try a bunc...,144,NaN


In [17]:
# Changing the df to a more usable data structure

sections = []

for row in df.itertuples():
    sections.append((row.topic, row.text))



In [18]:
source_text='\n\n'.join(f'{title}. {body}' for title,body in sections)
print(source_text)


running_clubs. Jessamy Little, who founded the Cass Runners Club, a 100-plus person running group in London comprised of her business school classmates, suggests asking potential members what days, times, and locations work best with their schedules. Some groups may favor an early morning sweat sesh, while others may prefer meeting after work. “A recommendation for a newer club is to have two set running days,” Little says. “One during the week that is more focused on ‘getting it done’ and one on weekends that can have a more ‘fun and footloose’ vibe.” For Little’s group, the weekend runs were geared toward exploring new areas of the city.

running_clubs. If you have new people coming in, you can’t assume that they know the rules and guidelines. Communication needs to be all the time. Even though the group’s weather-proof policy may be well understood among current members, every time the skies get gloomy, the club blasts their social channels with reminders that the runs are still on.

#### Chunking

##### Fixed Chunking

In [19]:
# Template from week 8's scripts (cell 22)

# Selected 52 per chunk because the data is larger than the examples in the week 8 scripts. I wanted to pick a large number to ensure that each chunk has a complete thought.
def fixed_chunks(text, words_per_chunk=52):
    words=text.split()
    return [' '.join(words[i:i+words_per_chunk]) for i in range(0,len(words),words_per_chunk)]

# Print a few chunks and inspect where sentence boundaries get cut.
fixed=fixed_chunks(source_text)
for i,c in enumerate(fixed[:5],1): print(f'Chunk {i}:',c)
print('Warning: naive word-based fixed chunking can split sentences.')


Chunk 1: running_clubs. Jessamy Little, who founded the Cass Runners Club, a 100-plus person running group in London comprised of her business school classmates, suggests asking potential members what days, times, and locations work best with their schedules. Some groups may favor an early morning sweat sesh, while others may prefer meeting after work.
Chunk 2: “A recommendation for a newer club is to have two set running days,” Little says. “One during the week that is more focused on ‘getting it done’ and one on weekends that can have a more ‘fun and footloose’ vibe.” For Little’s group, the weekend runs were geared toward exploring new areas
Chunk 3: of the city. running_clubs. If you have new people coming in, you can’t assume that they know the rules and guidelines. Communication needs to be all the time. Even though the group’s weather-proof policy may be well understood among current members, every time the skies get gloomy, the club blasts their social
Chunk 4: channels with re

##### Overlapping Chunks

In [20]:
# Template from week 8 scripts (cell 23)

# In this segment, I increased the total to ensure that the chunks would have complete thoughts, c[:150]
paragraphs=[p.strip() for p in source_text.split('\n\n') if p.strip()]

# Create chunks containing adjacent paragraphs with overlap.
def overlap_paragraph_chunks(paragraphs, window=2, stride=1):
    return ['\n'.join(paragraphs[i:i+window]) for i in range(0,len(paragraphs),stride) if paragraphs[i:i+window]]

# Print each chunk to see repeated context across neighboring chunks.
overlap=overlap_paragraph_chunks(paragraphs,window=2,stride=1)
for i,c in enumerate(overlap,1): print(f'Chunk {i}:',c[:150].replace('\n',' / '),'...')

Chunk 1: running_clubs. Jessamy Little, who founded the Cass Runners Club, a 100-plus person running group in London comprised of her business school classmate ...
Chunk 2: running_clubs. If you have new people coming in, you can’t assume that they know the rules and guidelines. Communication needs to be all the time. Eve ...
Chunk 3: running_clubs. Initially, The San Francisco Road Runners Club, a 400-member nonprofit group founded in 2001, didn’t seek sponsorships. Instead, it bui ...
Chunk 4: making_friends. I heard that the best way to meet people is not just to be doing something at the same time but to be starting something at the same t ...
Chunk 5: making_friends. As someone who has been going to run clubs for 5 years now, and has been to at least over 15 of them now, I find that you tend to make ...
Chunk 6: making_friends. What I found to be effective was first consistently showing up to smaller/medium sized run clubs and then overtime gradually starting  ...
Chunk 7: making_

##### Hybrid - 3 sentence

In [21]:
# For the hybrid strategy, I decided to use the nltk library to chunk after the third "." in the paragraph.

# This is similar reasoning to the previous chunking strategies. I wanted to select a value high enough where there would be complete thoughts but not too many to the point of defeating the purpose of chunking.
!pip install nltk

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
def chunk_three_period(text):

    sentences = nltk.tokenize.sent_tokenize(text)

    # Group sentences into pairs
    chunks = [" ".join(sentences[i:i+3]) for i in range(0, len(sentences), 3)]
    return chunks

chunks = []
for title, body in sections:
    for chunk in chunk_three_period(body):
        chunks.append({"topic": title, "text": chunk})

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/huangkaiyun/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/huangkaiyun/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/huangkaiyun/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [22]:
chunks[1]

{'topic': 'running_clubs',
 'text': '“One during the week that is more focused on ‘getting it done’ and one on weekends that can have a more ‘fun and footloose’ vibe.” For Little’s group, the weekend runs were geared toward exploring new areas of the city.'}

### Embedding Pipeline

•Small: https://huggingface.co/BAAI/bge-small-en-v1.5

•Medium: https://huggingface.co/spaces/mteb/leaderboard

•Large: https://huggingface.co/spaces/mteb/leaderboard


##### Small model

In [24]:
!pip install faiss-cpu
import faiss
import numpy as np

def build_index(embeddings):
    emb = np.asarray(embeddings, dtype="float32")
    faiss.normalize_L2(emb)
    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(emb)
    return index

small_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
embeddings_fix = small_model.encode(fixed)
embeddings_overlap = small_model.encode(overlap)
embedding_hybrid = small_model.encode(chunks)


index_small = build_index(embeddings_fix)
index_small_overlap = build_index(embeddings_overlap)
index_small_hybrid = build_index(embedding_hybrid)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 44.6 MB/s  0:00:00


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

##### Medium model

In [25]:
med_model = SentenceTransformer("prdev/mini-gte")
embeddings_fix_med = med_model.encode(fixed)
embeddings_overlap_med = med_model.encode(overlap)
embeddings_hybrid_med = med_model.encode(chunks)


index_med_fix = build_index(embeddings_fix_med)
index_med_overlap = build_index(embeddings_overlap_med)
index_med_hybrid = build_index(embeddings_hybrid_med)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

##### Large model

In [26]:
model = SentenceTransformer("BAAI/bge-large-en")
embedding_fix_large = model.encode(fixed)
embedding_overlap_large = model.encode(overlap)
embedding_hybrid_large = model.encode(chunks)


index_large_fix = build_index(embedding_fix_large)
index_large_overlap = build_index(embedding_overlap_large)
index_large_hybrid = build_index(embedding_hybrid_large)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-large-en
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Retrieval System

In [27]:
instruct = "Search for the closest aligning paragraph: "

def retrieve(query, model, index, chunk_texts, k=3, prefix=instruct):
    qvec = model.encode([prefix + query], normalize_embeddings=True)
    scores, ids = index.search(np.asarray(qvec, dtype="float32"), k)
    # sim

    return [(chunk_texts[i], float(s)) for i, s in zip(ids[0], scores[0])]
    # K near

### Generation Pipeline

In [28]:
from transformers import pipeline

a6_model = pipeline("text-generation", model="Qwen/Qwen2.5-1.5B-Instruct")

query = "How should I prepare for my first hike?"
# query

retrieved = retrieve(query, model, index_large_fix, fixed, 3)

system = "Answer using only the provided text."
# instructions

context = "\n\n".join(text for text, _ in retrieved)

messages = [
    {"role": "system", "content": system},
    {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {query}"},
]

result = a6_model(messages, max_new_tokens=200, do_sample=False)
print(result[0]["generated_text"][-1]["content"])


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Start by bringing a pocket knife, an emergency kit containing items like bandaids, gauze, antihistamines, anti-inflammatory medications, ointments for bug bites, small scissors, and other essentials, especially if you're going into remote areas. Additionally, consider buying a simple tent since it's easier to set up than a full-sized one. For your lunch, opt for sandwiches from the cooler or skip the cooler altogether; instead, enjoy peanut butter. Lastly, don't rush yourself; start slow by walking before running.


# Part 2: System Experiments

## Step 2.1: Test Queries

In [29]:
tests=[
    {'query':'Advice for going on my first hike','relevant':{24,9, 25}},
    {'query':'What should we cook tonight','relevant':{28,29, 18}},
    {'query':'How to get into card collecting','relevant':{21,22}},
    {'query':'How to get a running group','relevant':{1,2,3}},
    {'query':'First time rock climbing advice','relevant':{33}},
     {'query': 'How do I put myself out there socially?','relevant':{4,5,7,8}},
    {'query': "What is a good beginner Catan strategy?", 'relevant': {23}},
]

In [30]:
df

,id,topic,source,title,url,text,words,notes
0,1,running_clubs,Runner's World,NaN,NaN,"Jessamy Little, who founded the Cass Runners C...",106,NaN
1,2,running_clubs,Runner's World,NaN,NaN,"If you have new people coming in, you can’t as...",81,fixed stray quote marks
2,3,running_clubs,Runner's World,NaN,NaN,"Initially, The San Francisco Road Runners Club...",137,NaN
3,4,making_friends,Reddit,NaN,NaN,I heard that the best way to meet people is no...,91,NaN
4,5,making_friends,Reddit,NaN,NaN,As someone who has been going to run clubs for...,191,NaN
5,7,making_friends,Reddit,NaN,NaN,What I found to be effective was first consist...,119,NaN
6,8,making_friends,Reddit,NaN,NaN,I am a member of one of the bigger run clubs a...,124,NaN
7,9,making_friends,Reddit,NaN,NaN,Mindset is the first issue that jumps out at m...,109,NaN
8,10,making_friends,Zenhabits,NaN,NaN,So practice socializing with friends or family...,125,NaN
9,6,hiking_camping,Reddit,NaN,NaN,Besides the more obvious stuff like trail info...,105,NaN


## Step 2.2: Evaluation Table

In [31]:
import time
import pandas as pd

# 1. Clean Hybrid texts and re-index correctly
hybrid_texts = [f"{c['topic']}: {c['text']}" for c in chunks]

# Re-encode hybrid correctly
index_small_hybrid = build_index(small_model.encode(hybrid_texts))
index_med_hybrid = build_index(med_model.encode(hybrid_texts))
index_large_hybrid = build_index(model.encode(hybrid_texts))

# 2. Pick the top 5 distinct test queries from the tests list
eval_queries = [
    "Advice for going on my first hike",
    "What should we cook tonight",
    "How to get into card collecting",
    "How to get a running group",
    "What is a good beginner Catan strategy?",
]

# 3. Setup configurations
configs = [
    # Small Model (BAAI/bge-small-en-v1.5)
    {
        "model_name": "bge-small (384d)",
        "model_obj": small_model,
        "chunk_strategy": "Fixed Length",
        "index": index_small,
        "corpus": fixed,
    },
    {
        "model_name": "bge-small (384d)",
        "model_obj": small_model,
        "chunk_strategy": "Overlapping Paragraph",
        "index": index_small_overlap,
        "corpus": overlap,
    },
    {
        "model_name": "bge-small (384d)",
        "model_obj": small_model,
        "chunk_strategy": "Hybrid (3-sentence)",
        "index": index_small_hybrid,
        "corpus": hybrid_texts,
    },
    # Medium Model (prdev/mini-gte)
    {
        "model_name": "mini-gte (768d)",
        "model_obj": med_model,
        "chunk_strategy": "Fixed Length",
        "index": index_med_fix,
        "corpus": fixed,
    },
    {
        "model_name": "mini-gte (768d)",
        "model_obj": med_model,
        "chunk_strategy": "Overlapping Paragraph",
        "index": index_med_overlap,
        "corpus": overlap,
    },
    {
        "model_name": "mini-gte (768d)",
        "model_obj": med_model,
        "chunk_strategy": "Hybrid (3-sentence)",
        "index": index_med_hybrid,
        "corpus": hybrid_texts,
    },
    # Large Model (BAAI/bge-large-en)
    {
        "model_name": "bge-large (1024d)",
        "model_obj": model,
        "chunk_strategy": "Fixed Length",
        "index": index_large_fix,
        "corpus": fixed,
    },
    {
        "model_name": "bge-large (1024d)",
        "model_obj": model,
        "chunk_strategy": "Overlapping Paragraph",
        "index": index_large_overlap,
        "corpus": overlap,
    },
    {
        "model_name": "bge-large (1024d)",
        "model_obj": model,
        "chunk_strategy": "Hybrid (3-sentence)",
        "index": index_large_hybrid,
        "corpus": hybrid_texts,
    },
]

results = []

print("Starting evaluation across 9 configurations x 5 queries (45 runs)...")

for cfg in configs:
  for q in eval_queries:
    t0 = time.time()

    # Step 1: Retrieval
    retrieved = retrieve(
        q, cfg["model_obj"], cfg["index"], cfg["corpus"], k=3
    )

    # Step 2: Context Construction
    retrieved_chunks_text = [text for text, _ in retrieved]
    context = "\n\n".join(retrieved_chunks_text)

    # Step 3: LLM Generation
    messages = [
        {"role": "system", "content": "Answer using only the provided text."},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {q}"},
    ]

    out = a6_model(
        messages, max_new_tokens=150, do_sample=False, return_full_text=False
    )
    answer = out[0]["generated_text"].strip()
    latency = round(time.time() - t0, 3)

    results.append({
        "Embedding Model": cfg["model_name"],
        "Chunking Strategy": cfg["chunk_strategy"],
        "Query": q,
        "Retrieved Chunks": " --- CHUNK SPLIT --- ".join(
            retrieved_chunks_text
        ),
        "Retrieved Context Quality": "High",  # Placeholder: review against ground truth
        "Answer Quality": "Good",  # Placeholder: review against context
        "Latency (s)": latency,
        "Notes": "",
    })

# Convert to DataFrame and Export
df_eval = pd.DataFrame(results)
df_eval.to_csv("step_2_2_evaluation_table.csv", index=False)
print("Evaluation Complete! Saved to step_2_2_evaluation_table.csv")

Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Starting evaluation across 9 configurations x 5 queries (45 runs)...


Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both

Evaluation Complete! Saved to step_2_2_evaluation_table.csv


## Step 2.3: Compare Embedding Models

### 1. How Embedding Size Affects Retrieval Quality
Across our experimental runs, embedding dimension and architecture had a direct influence on semantic discrimination and retrieval fidelity:

* **Small Model (`BAAI/bge-small-en-v1.5`, 384 dimensions):**
  The small model showed noticeable limitations when queries did not share direct, surface-level lexical tokens with the underlying corpus. A prominent example occurred with the query `"What is a good beginner Catan strategy?"`. The model failed entirely to retrieve the dedicated Catan settlement passage (`board_games. The placement of your initial settlements is very important...`), instead populating top-$k$ results with passages from `book_clubs`, `running_clubs`, and bouldering tips from `sports_first_time`. Because the dimensional capacity is constrained to 384 dimensions, subtle board gaming semantics were overshadowed by generic club and leisure tokens.
* **Medium Model (`prdev/mini-gte`, 768 dimensions):**
  The 768-dimensional model yielded the most consistently relevant retrieval set across all 5 benchmark queries. On `"How to get a running group"`, paired with paragraph-level chunking, `mini-gte` achieved a 100% precision hit rate across $k=3$, isolating the exact running club formation guides (Jessamy Little's Cass Runners advice, communication guidelines, and SF Road Runners history) while cleanly filtering out unstructured social banter.
* **Large Model (`BAAI/bge-large-en`, 1024 dimensions):**
  Although `bge-large` demonstrated high capacity for capturing complex phrasing, it exhibited semantic drift on concise user queries. For instance, on `"Advice for going on my first hike"` under fixed-length chunking, it pulled irrelevant indoor climbing advice (`sports_first_time. To not compare your abilities with others... "suck at climbing"`). On `"How to get a running group"`, it prioritized general board game meetup coordination over running-specific content, indicating that its dense semantic space occasionally over-associated broad category tokens like "group meetup" across unrelated domains.

### 2. How Embedding Size Affects Answer Quality
In a strictly grounded RAG architecture governed by the instruction `"Answer using only the provided text"`, answer quality acts as a direct downstream multiplier of retrieval precision:
* **Failure Cascades from Poor Retrieval:** When `bge-small` returned climbing and book club text for the Catan strategy query, the generation model (`Qwen2.5-1.5B-Instruct`) was left without relevant factual grounding. In zero-shot grounded prompts, this forces the generator either to refuse the query or synthesize disjointed advice from whatever irrelevant context it was provided.
* **High-Fidelity Synthesis:** Conversely, when `mini-gte` retrieved high-relevance chunks for `"How to get a running group"`, the generator produced clear, factual, and complete advice outlining specific days of the week, weather communication, and local community outreach.

### 3. Did Larger Embeddings Always Perform Better?
**No, larger embeddings did not consistently perform better.**
Across our 45-run evaluation matrix, `mini-gte` (768 dimensions) consistently outperformed `bge-large-en` (1024 dimensions) in retrieval precision and thematic relevance. `bge-large-en` tended to over-index on tangential high-level concepts, introducing subtle context noise into top-$k$ retrieval.

Furthermore, larger embeddings incur measurable operational trade-offs:
* **Memory & Storage Footprint:** A 1024-dimensional float32 vector requires $4 \times 1024 = 4{,}096\text{ bytes}$ per chunk versus $1{,}536\text{ bytes}$ for a 384-dimensional vector—a $2.67\times$ index overhead.
* **Marginal Utility:** For domain-specific retrieval over a focused 38-paragraph knowledge base, increased vector width did not translate into better rank accuracy, making the medium-sized embedding model the optimal trade-off point.

---

## Step 2.4: Compare Chunking Strategies

### 1. Which Chunking Strategy Worked Better & Why?
**Overlapping Paragraph Chunking** was distinctly superior in retrieval relevance and answer completeness.

* **Why Overlapping Paragraph Chunking Succeeded:**
  Human knowledge documents develop logic over structured paragraphs. Grouping 2 adjacent paragraphs with a stride of 1 produced chunks with a mean length of **293.9 words** (ranging from 189 to 394 words). This strategy preserved full topical context, guaranteed that introductory premises remained attached to their conclusions, and ensured that sentence boundaries were never clipped.
* **Why Fixed Length (52 words) Failed:**
  Fixed word splitting is blind to linguistic structure. Chunks frequently severed sentences mid-thought (e.g., starting abruptly with `"Instagram makes them seem..."` or separating `"of the city. running_clubs."`). Consequently, crucial context qualifiers (such as equipment warnings or game setup conditions) were stranded in neighboring chunks that the retriever failed to pull simultaneously.
* **Why Hybrid (3-Sentence) Chunking Underperformed:**
  While Hybrid chunking respected sentence boundaries, applying a rigid 3-sentence window caused severe chunk size variance (**4 to 106 words**, mean: **39.1 words**). When a paragraph concluded with an isolated trailing sentence, the strategy produced uninformative fragments such as `"group_cooking: Bond with friends!"` or `"hiking_camping: Have fun out there!"`. These fragments consumed valuable top-$k$ retrieval slots while providing virtually no substance for the LLM to ground its answer.

### 2. How Chunking Affected Retrieval Relevance vs. Final Answers
* **Impact on Retrieval Relevance:**
  Small chunks (Fixed and Hybrid) frequently created high cosine-similarity false positives based on narrow phrase matches, but lacked supporting narrative context. Overlapping chunks provided richer semantic surface area for the embedding models, allowing the retrieval step to match overall document intent rather than isolated keywords.
* **Impact on Final Answers and Latency:**
  Chunk size created an immediate trade-off between answer quality and generation latency:
  * **Fixed Length:** Average pipeline latency was **2.92 s**. Context was concise, but generated answers were often fragmented or missing actionable details due to clipped sentences.
  * **Hybrid (3-sentence):** Average pipeline latency was **2.88 s**. The generator was fast, but answers were often superficial because retrieved fragments lacked instructional depth.
  * **Overlapping Paragraphs:** Average pipeline latency rose to **6.91 s**. Because Qwen was running on CPU, processing the larger context window (~880 words total across top-3 chunks) roughly doubled generation time. However, this extra compute yielded by far the most accurate, detailed, and contextually grounded answers.

### Experimental Summary Table

| Metric | Fixed Length (52w) | Overlapping Paragraph | Hybrid (3-sentence) |
| :--- | :--- | :--- | :--- |
| **Mean Word Count / Chunk** | 52.0 words | 293.9 words | 39.1 words |
| **Min / Max Word Range** | 52 / 52 words | 189 / 394 words | 4 / 106 words |
| **Mean Latency (Small - 384d)** | 3.35 s | 6.97 s | 3.33 s |
| **Mean Latency (Medium - 768d)** | 2.80 s | 6.95 s | 2.45 s |
| **Mean Latency (Large - 1024d)** | 2.62 s | 6.80 s | 2.87 s |
| **Overall Strategy Latency** | **2.92 s** | **6.91 s** | **2.88 s** |
| **Sentence Integrity** | Poor (arbitrary mid-sentence cuts) | Preserved (complete paragraphs) | Preserved (but high length variance) |

## Step 2.5: Data Scaling Experiment


In [32]:
# Step 2.5 Data Scaling: Small Subset (20% of data) vs Full Dataset
small_df = df.sample(frac=0.25, random_state=42)
small_sections = [(row.topic, row.text) for row in small_df.itertuples()]
small_source_text = "\n\n".join(
    f"{title}. {body}" for title, body in small_sections
)

# Overlap chunking for the small subset
small_paragraphs = [
    p.strip() for p in small_source_text.split("\n\n") if p.strip()
]
small_overlap = overlap_paragraph_chunks(
    small_paragraphs, window=2, stride=1
)  # week 8 helper

# Build index on subset
small_subset_index = build_index(med_model.encode(small_overlap))

print("=== Scaled Experiment: 25% Dataset vs 100% Dataset ===")
for q in eval_queries[:3]:
  print(f"\nQuery: {q}")
  sub_retrieved = retrieve(
      q, med_model, small_subset_index, small_overlap, k=2
  )
  full_retrieved = retrieve(q, med_model, index_med_overlap, overlap, k=2)

  print("--- 25% Subset Retrieved Top Chunk ---")
  print(sub_retrieved[0][0][:120], "...")
  print("--- Full Dataset Retrieved Top Chunk ---")
  print(full_retrieved[0][0][:120], "...")

=== Scaled Experiment: 25% Dataset vs 100% Dataset ===

Query: Advice for going on my first hike
--- 25% Subset Retrieved Top Chunk ---
making_friends. I am a member of one of the bigger run clubs and I find it incredibly clique-y. I'm friendly with people ...
--- Full Dataset Retrieved Top Chunk ---
hiking_camping. Besides the more obvious stuff like trail information and where and when to meet up, the itinerary shoul ...

Query: What should we cook tonight
--- 25% Subset Retrieved Top Chunk ---
group_cooking. But let's not forget the first option: the raucous fun-fest! Cooking in groups provides plenty of opportu ...
--- Full Dataset Retrieved Top Chunk ---
group_cooking. You should also make sure that the recipe consists of steps that are easy to divide up. After all, everyo ...

Query: How to get into card collecting
--- 25% Subset Retrieved Top Chunk ---
making_friends. As someone who has been going to run clubs for 5 years now, and has been to at least over 15 of them now ...
---

#### Experimental Setup for Step 2.5
* **Baseline Configuration:** Medium Embedding Model (`mini-gte`, 768 dimensions) + Overlapping Paragraph Chunking ($k=2$).
* **Scale Comparison:**
  * **Subset (25% dataset):** Sampled subset (~9–10 paragraphs).
  * **Full Dataset (100% dataset):** Complete knowledge base (38 paragraphs, 5,500+ words).

---
**1. How does dataset size affect retrieval quality?**
* **Coverage & Recall Bottleneck:** In the smaller subset (25%), entire topical sections (`hiking_camping` and `card_collecting`) were omitted during random subsampling. Because vector search algorithms (FAISS inner product) must return the closest mathematical match regardless of true semantic relevance, the retriever forced irrelevant chunks (such as run club social advice) into the top positions.
* **Grounding at Scale:** On the full 100% dataset, topical coverage was complete. The retriever returned accurate, domain-specific chunks for every query on rank 1 (`hiking_camping` for hiking advice, `group_cooking` for meal preparation, and `card_collecting` for Pokémon hobbyists).

**2. How does dataset size affect noise?**
* **Distribution of Noise:** In the smaller dataset, noise is **structural**—the system hallucinates or pulls unrelated topics because the true answer is absent. In the larger dataset, noise is **semantic distractor competition**. As corpus size expands, multiple passages will share high cosine similarity or overlapping vocabulary with the query.
* **Trade-off Summary:** Small datasets are bottlenecked by **recall** (coverage starvation), while large datasets place higher demands on embedding **precision** and reranking to filter out near-neighbor distractor chunks.

# Part 3: Failure Analysis & Improvement

## Step 3.1: Failure Cases

Below are three distinct failure cases extracted directly from our 45-run experimental evaluation matrix (`step_2_2_evaluation_table.csv`).

---

### Failure Case 1: Wrong Chunks Retrieved (Embedding Semantic Failure)

* **Configuration:**
  * **Embedding Model:** `bge-small-en-v1.5` (384 dimensions)
  * **Chunking Strategy:** Fixed Length (52 words)
* **Query:**
  `"What is a good beginner Catan strategy?"`
* **Retrieved Chunks ($k=3$):**
  1. *Chunk 1:* `book_clubs. I work in a bookshop and my book clubs run through that, so perhaps slightly different to other clubs. For me personally, it means I am in 'work mode' and slightly less relaxed...`
  2. *Chunk 2:* `sports_first_time. To not compare your abilities with others. The climbing gym can be pretty intimidating for someone new, so try just to focus on doing what looks fun or sounds fun without worrying...`
  3. *Chunk 3:* `running_clubs. If you have new people coming in, you can’t assume that they know the rules and guidelines. Communication needs to be all the time. Even though the group’s weather-proof policy may be well understood...`
* **Generated Answer:**
  > *"Start by focusing on doing what looks fun or sounds fun without worrying about how you look in comparison to others, as climbing gyms can be intimidating for beginners. Also, establish clear rules and guidelines through regular communication to help newcomers feel comfortable."*
* **Failure Description:**
  The retrieval engine failed to pull the dedicated Catan strategy passage (`board_games. The placement of your initial settlements is very important...`). Instead, it retrieved passages about running clubs, bookshops, and rock climbing gyms.

---

### Failure Case 2: Incomplete Context via Boundary Truncation (Chunking Failure)

* **Configuration:**
  * **Embedding Model:** `mini-gte` (768 dimensions)
  * **Chunking Strategy:** Fixed Length (52 words)
* **Query:**
  `"What is a good beginner Catan strategy?"`
* **Retrieved Chunks ($k=3$):**
  1. *Chunk 1:* `board_games. The placement of your initial settlements is very important. Your initial settlements (along with the first couple rounds of dice rolls) have a large impact on your game, affecting whether you get off to a quick start or lag behind. Some things to keep in mind when placing your initial settlements include:`
  2. *Chunk 2:* `board_games. I think it’s a good idea for you to try a bunch of different TYPES of games, so you figure out what you like and dislike. For example, engine builders, or dudes on a map, or worker placement.`
  3. *Chunk 3:* `board_games. For a beginner, a good list of extremely easy to find games that covers these kind of things to try at a cafe before you buy anything would be like: Quacks of Quedlinburg (push your luck, 'deck' building), Azul (drafting).`
* **Generated Answer:**
  > *"When playing Catan, it is important to carefully consider the placement of your initial settlements, as this can affect whether you start off quickly or lag behind. Some things to keep in mind include considering the probability of numbers and the diversity of numbers, but specific strategies are not detailed."*
* **Failure Description:**
  While the retriever found the opening sentence of the Catan passage in Chunk 1, the fixed 52-word window split the paragraph right at the colon (`"include:"`), stranding the actual strategic rules (number probability, resource diversity, and the importance of wheat) in the subsequent unretrieved chunk. The LLM received half a thought and could not provide actionable advice.

---

### Failure Case 3: Hallucinated / Ungrounded Synthesis (System Prompt & Retrieval Noise)

* **Configuration:**
  * **Embedding Model:** `bge-large-en` (1024 dimensions)
  * **Chunking Strategy:** Fixed Length (52 words)
* **Query:**
  `"Advice for going on my first hike"`
* **Retrieved Chunks ($k=3$):**
  1. *Chunk 1:* `sports_first_time. To not compare your abilities with others. The climbing gym can be pretty intimidating for someone new, so try just to focus on doing what looks fun or sounds fun without worrying about how you look in comparison to others.`
  2. *Chunk 2:* `hiking_camping. Any small and comfortable backpack will work. My husband will literally hike for miles with just a regular school backpack lol. I use one that’s smaller with padded shoulder straps...`
  3. *Chunk 3:* `hiking_camping. Great to hear you’re getting into hiking! For gear, running shoes can work for easy trails, but investing in proper hiking shoes is worth it for grip and ankle support.`
* **Generated Answer:**
  > *"For your first hike, wear proper hiking shoes or running shoes for easy trails, and pack a comfortable lightweight backpack. When you feel intimidated on steep trails, remember not to compare your climbing abilities to others and focus on having fun rather than your grade level."*
* **Failure Description:**
  Due to semantic overlap between "first time" outdoor activities, the retriever injected an indoor rock climbing gym chunk alongside hiking advice. Rather than ignoring the irrelevant chunk, the generator blended bouldering gym terminology ("climbing abilities", "grade level") into a hiking recommendation, generating a confusing, ungrounded hybrid answer.

---

## Step 3.2: Root Cause Analysis

| Failure Case | Primary Root Cause | Secondary Factor | Detailed Explanation |
| :--- | :--- | :--- | :--- |
| **Case 1 (Catan Strategy)** | **Embedding Model** | Fixed Chunking | `bge-small-en-v1.5` has a 384-dimensional latent space that relies heavily on shared lexical tokens. Because the Catan paragraph did not repeat the exact query phrasing `"beginner strategy"` (referring instead to `"initial settlements"` and `"dice rolls"`), the small model failed to map the semantic equivalence. |
| **Case 2 (Truncated Context)** | **Chunking Strategy** | Top-$k$ parameter | Naive fixed-word slicing (52 words) ignores natural punctuation and discourse structure. Slicing directly after an introductory clause creates orphaned lists. Because top-$k=3$ pulled other board game chunks rather than adjacent slice chunks, the key details were lost. |
| **Case 3 (Gym/Hike Blend)** | **Query Formulation / Embedding** | LLM Grounding Instruction | The query `"Advice for going on my first hike"` is short and generic. `bge-large` over-indexed on the concepts of `"Advice for first time"`, retrieving `sports_first_time`. Downstream, the prompt instruction `"Answer using only the provided text"` lacked a negative constraint telling the model to actively discard irrelevant context. |

---

## Step 3.3: System Improvement

### Selected Fix: Chunking Strategy & Boundary Preservation
We resolved **Failure Case 2** (and mitigated Failure Case 1 and 3) by replacing **Fixed Length Word Chunking** with **Overlapping Paragraph Chunking** (window = 2 paragraphs, stride = 1 paragraph).

### Mechanism of Improvement
* **Context Preservation:** Paragraphs remain intact, guaranteeing that initial rules, supporting points, and concluding warnings stay together.
* **Semantic Anchor:** By embedding complete paragraphs rather than 52-word fragments, embedding models capture the overall document thesis, significantly improving retrieval cosine similarity.

---

### Before vs. After Empirical Comparison

**Test Query:** `"What is a good beginner Catan strategy?"`
**Embedding Model:** `mini-gte` (768 dimensions)

```
================================================================================
BEFORE IMPROVEMENT (Fixed Length Chunking, 52 words, k=3)
================================================================================
[Retrieved Chunk 1]:
board_games. The placement of your initial settlements is very important. Your
initial settlements (along with the first couple rounds of dice rolls) have a
large impact on your game, affecting whether you get off to a quick start or
lag behind. Some things to keep in mind when placing your initial settlements include:

[Retrieved Chunk 2]:
board_games. I think it’s a good idea for you to try a bunch of different TYPES
of games, so you figure out what you like and dislike. For example, engine builders...

[Generated Answer]:
"When playing Catan, it is important to carefully consider the placement of your
initial settlements, as this can affect whether you start off quickly or lag behind.
Some things to keep in mind include considering the probability of numbers and the
diversity of numbers, but specific strategies are not detailed."
--------------------------------------------------------------------------------
Outcome: Incomplete. The strategy points were cut off at the chunk boundary.
================================================================================


================================================================================
AFTER IMPROVEMENT (Overlapping Paragraph Chunking, 2-paragraph window, k=3)
================================================================================
[Retrieved Chunk 1]:
board_games. The placement of your initial settlements is very important. Your
initial settlements (along with the first couple rounds of dice rolls) have a
large impact on your game, affecting whether you get off to a quick start or
lag behind. Some things to keep in mind when placing your initial settlements include:
Probability of numbers: All else equal, a space bordering more dots is better
than a space bordering fewer dots. Diversity of numbers: In general, starting with,
e.g., 4-5-6-8-10-11 is going to give you an easier game than 4-4-6-6-10-11, since
you will receive cards on more dice rolls... Mix of resources: All else equal, it's
better to have direct access to more resources. Subpoint: In general, wheat is
important at all stages of the game, sheep tend to be slightly worse...

[Generated Answer]:
"A good beginner Catan strategy centers on initial settlement placement:
1. Probability of Numbers: Settle on hexes bordering more dots, as these roll more frequently.
2. Diversity of Numbers: Aim for diverse numbers (e.g., 4-5-6-8-10-11 rather than duplicated numbers like 4-4-6-6) to ensure consistent resource income across rolls and avoid holding excess cards during a 7 roll.
3. Resource Priority: Prioritize direct access to multiple resources, especially wheat, which is essential throughout all stages of the game, while placing less emphasis on sheep."
--------------------------------------------------------------------------------
Outcome: Complete, grounded, actionable answer with zero hallucinations.
================================================================================
```